# JOINTS 2026 EDA Notebook

Objective: Explore the competition data and the external datasets to understand how daily ticket sales on D4-D10 behave for each film and cinema pair, the target scored with MASE.

# Import modules and libraries

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import datasets

In [ ]:
_DATA_DIR_CANDIDATES = [
    "../data", "data", "../../data",
    *glob.glob("/kaggle/input/*"), *glob.glob("/kaggle/input/**/", recursive=True),
]
DATA_DIR = next((d for d in _DATA_DIR_CANDIDATES if d and os.path.exists(os.path.join(d, "test_history.csv"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("test_history.csv not found in any candidate DATA_DIR - set it manually.")
print("DATA_DIR =", os.path.abspath(DATA_DIR))

train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["date_show"])
test_hist = pd.read_csv(f"{DATA_DIR}/test_history.csv", parse_dates=["date_show"])
test = pd.read_csv(f"{DATA_DIR}/test.csv", parse_dates=["date_show"])
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
movies = pd.read_csv(f"{DATA_DIR}/movies.csv")
holidays = pd.read_csv(f"{DATA_DIR}/holidays.csv", parse_dates=["date"])
prices = pd.read_csv(f"{DATA_DIR}/ticket_prices.csv")

### External datasets
Calendar files built by external_add.ipynb in the external/ folder.

In [ ]:
_EXTERNAL_DIR_CANDIDATES = ["../external", "external", "../../external", *glob.glob("/kaggle/input/*")]
EXTERNAL_DIR = next((d for d in _EXTERNAL_DIR_CANDIDATES if os.path.exists(os.path.join(d, "cuti_bersama.csv"))), None)
if EXTERNAL_DIR is None:
    raise FileNotFoundError("cuti_bersama.csv not found in any candidate EXTERNAL_DIR - set it manually.")
print("EXTERNAL_DIR =", os.path.abspath(EXTERNAL_DIR))

cuti = pd.read_csv(f"{EXTERNAL_DIR}/cuti_bersama.csv", parse_dates=["date"])
ramadan = pd.read_csv(f"{EXTERNAL_DIR}/ramadan.csv", parse_dates=["date"])
school = pd.read_csv(f"{EXTERNAL_DIR}/school_holidays.csv", parse_dates=["date"])

# EDA

### Dataset summary
Size, date range, missing values and duplicates of every file.

In [ ]:
DATASETS = {"train": train, "test_history": test_hist, "test": test, "sample_sub": sample_sub,
            "movies": movies, "holidays": holidays, "prices": prices}
def describe_file(df):
    dates = df["date_show"] if "date_show" in df else df.get("date")
    return {
        "rows": len(df), "cols": df.shape[1],
        "missing": int(df.isna().sum().sum()), "duplicates": int(df.duplicated().sum()),
        "date_min": dates.min().date() if dates is not None else "-",
        "date_max": dates.max().date() if dates is not None else "-",
    }

pd.DataFrame({name: describe_file(df) for name, df in DATASETS.items()}).T

In [ ]:
train.describe()

### Train and test overlap
How many films, cinemas and cities in test also appear in train, since this decides which features can be learned from train.

In [ ]:
def overlap(col):
    tr, te = set(train[col]), set(test[col])
    return {"train": len(tr), "test": len(te), "overlap": len(tr & te), "test_only": len(te - tr)}

meta_titles = set(movies["original_title"])
all_titles = set(train["movie_title"]) | set(test_hist["movie_title"])
print(pd.DataFrame({c: overlap(c) for c in ["movie_title", "cinema_ids", "city_name"]}).T)
print(f"\nfilm yang ada di movies.csv: {len(all_titles & meta_titles)} / {len(all_titles)}")
print("film test tanpa metadata:", sorted(set(test["movie_title"]) - meta_titles))
print("kota tanpa harga tiket:", sorted((set(train["city_name"]) | set(test_hist["city_name"])) - set(prices["city_name"])))

### Film formats
Some titles are another format of the same film (3D, IMAX). These titles are missing from movies.csv, so their metadata has to come from the base title.

In [ ]:
FORMAT_PATTERN = r"\s*\((?:IMAX )?[23]D\)\s*$"
formats = pd.DataFrame({
    name: df.drop_duplicates("movie_title")["movie_title"].str.extract(r"\(((?:IMAX )?[23]D)\)\s*$")[0].fillna("regular").value_counts()
    for name, df in {"train": train, "test": test}.items()
}).fillna(0).astype(int)
base_titles = pd.Series(sorted(all_titles)).str.replace(FORMAT_PATTERN, "", regex=True)
print(formats)
print(f"\nfilm dengan metadata setelah judul format dihapus: {base_titles.isin(meta_titles).sum()} / {len(base_titles)}")
print("masih tanpa metadata:", sorted(set(base_titles[~base_titles.isin(meta_titles)]))[:20])

### Daily ticket distribution
Ticket counts are shown on a log scale because they differ hugely between films and cinemas.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(np.log1p(train["total_ticket"]), bins=50, alpha=0.6, density=True, label="train")
ax[0].hist(np.log1p(test_hist["total_ticket"]), bins=50, alpha=0.6, density=True, label="test_history")
ax[0].set_title("log1p(total_ticket)")
ax[0].legend()
ax[1].hist(train["occupation_rate"], bins=50)
ax[1].set_title("occupation_rate train")
ax[2].hist(np.log1p(train["total_show"]), bins=50)
ax[2].set_title("log1p(total_show) train")
plt.show()

### National tickets per day
Total tickets of all films per date for train and test_history, with national holidays marked in red, to see busy seasons and holiday effects.

In [ ]:
daily = pd.concat([train, test_hist]).groupby("date_show")["total_ticket"].sum()
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(daily.index, daily.values, lw=1)
for d in holidays.loc[holidays["holiday_tipe"] == "holiday", "date"]:
    ax.axvline(d, color="red", alpha=0.3)
ax.axvline(test_hist["date_show"].min(), color="black", ls="--", label="mulai test_history")
ax.set_ylabel("total tiket")
ax.legend()
plt.show()

### Day and holiday effects
Average tickets per film-cinema-day by day of week and day type.

In [ ]:
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
cal = train.merge(holidays.rename(columns={"date": "date_show"}), on="date_show", how="left")
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
cal.groupby(cal["date_show"].dt.day_name())["total_ticket"].mean().reindex(DAY_ORDER).plot(kind="bar", ax=ax[0], title="per hari")
cal.groupby(["day_tipe", "holiday_tipe"])["total_ticket"].mean().plot(kind="bar", ax=ax[1], title="per tipe hari dan libur", rot=30)
plt.tight_layout()
plt.show()
holidays[holidays["holiday_tipe"] == "holiday"]

### Cinemas per day for each film
Films often have a few preview days in a handful of cinemas before the wide release; this pattern is used to set D1 in train.

In [ ]:
clusters_per_day = train.groupby(["movie_title", "date_show"])["cinema_ids"].nunique()
first_date = train.groupby("movie_title")["date_show"].min()
max_clusters = clusters_per_day.groupby(level=0).max()
examples = max_clusters[first_date > first_date.min()].nlargest(4).index

fig, ax = plt.subplots(figsize=(12, 4))
for title in examples:
    s = clusters_per_day[title]
    ax.plot((s.index - s.index.min()).days, s.values, marker="o", label=title)
ax.set_xlim(0, 20)
ax.set_xlabel("hari sejak transaksi pertama")
ax.set_ylabel("jumlah klaster")
ax.legend()
plt.show()

### D1-D10 decay curve
Tickets per day divided by the pair's D1-D3 average (the MASE scale), with train D1 = first day the film plays in at least 50% of its max cinema count. This is the shape of the target.

In [ ]:
eda_d1 = (
    clusters_per_day[clusters_per_day >= 0.5 * clusters_per_day.groupby(level=0).transform("max")]
    .reset_index().groupby("movie_title")["date_show"].min()
)
eda_d1 = eda_d1[(first_date[eda_d1.index] > train["date_show"].min())
                & (eda_d1 + pd.Timedelta(days=9) <= train["date_show"].max())]

curve = train[train["movie_title"].isin(eda_d1.index)].copy()
curve["day"] = (curve["date_show"] - curve["movie_title"].map(eda_d1)).dt.days + 1
curve = curve[curve["day"].between(1, 10)].pivot_table(
    index=["movie_title", "cinema_ids"], columns="day", values="total_ticket", fill_value=0
).reindex(columns=range(1, 11), fill_value=0)
curve = curve[curve[3] > 0]
ratio = curve.div(curve[[1, 2, 3]].mean(axis=1), axis=0)

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(ratio.columns, ratio.median(), marker="o", label="median")
ax[0].fill_between(ratio.columns, ratio.quantile(0.25), ratio.quantile(0.75), alpha=0.3, label="IQR")
ax[0].axvline(3.5, color="black", ls="--")
ax[0].set_xlabel("hari ke-")
ax[0].set_ylabel("tiket / rata-rata D1-D3")
ax[0].legend()
ax[1].boxplot([ratio[d].clip(upper=3) for d in range(4, 11)], tick_labels=[f"D{d}" for d in range(4, 11)], showfliers=False)
ax[1].set_title("sebaran rasio target D4-D10")
plt.show()
print(f"film: {len(eda_d1)} | pasangan: {len(ratio)}")
print("porsi target nol per horizon:\n", (curve[list(range(4, 11))] == 0).mean().round(3).to_dict())

### D1 weekday and the curve
Median curve grouped by the weekday of D1, since where the weekend falls inside D1-D10 shifts the ups and downs. D1 weekdays with fewer than 30 pairs are hidden.

In [ ]:
d1_dow = ratio.index.get_level_values("movie_title").map(eda_d1).dayofweek
fig, ax = plt.subplots(figsize=(12, 4))
for dow, grp in ratio.groupby(d1_dow):
    if len(grp) < 30:
        continue
    ax.plot(grp.columns, grp.median(), marker="o", label=f"{DAY_ORDER[dow]} (n={len(grp)})")
ax.axvline(3.5, color="black", ls="--")
ax.set_xlabel("hari ke-")
ax.set_ylabel("median tiket / rata-rata D1-D3")
ax.legend()
plt.show()

### Test structure
Compares the D1 weekday and the D1-D3 scale of train and test pairs, to look for distribution shifts that could explain the CV-LB gap.

In [ ]:
test_d1 = test.groupby("movie_title")["date_show"].min() - pd.Timedelta(days=3)
test_pairs = test[["movie_title", "cinema_ids"]].drop_duplicates()
th = test_hist.merge(test_pairs, on=["movie_title", "cinema_ids"])
test_scale = th.groupby(["movie_title", "cinema_ids"])["total_ticket"].sum() / 3
train_scale = curve[[1, 2, 3]].mean(axis=1)

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
pd.DataFrame({
    "train": eda_d1.dt.day_name().value_counts(normalize=True),
    "test": test_d1.dt.day_name().value_counts(normalize=True),
}).reindex(DAY_ORDER).plot(kind="bar", ax=ax[0], title="hari D1 per film")
ax[1].hist(np.log1p(train_scale), bins=50, alpha=0.6, density=True, label="train")
ax[1].hist(np.log1p(test_scale), bins=50, alpha=0.6, density=True, label="test")
ax[1].set_title("log1p(scale) per pasangan")
ax[1].legend()
ax[2].hist(test_pairs.groupby("movie_title").size(), bins=30)
ax[2].set_title("jumlah klaster per film test")
plt.tight_layout()
plt.show()
print(f"test: {test_d1.size} film | {len(test_pairs)} pasangan | {len(test)} baris")
print("baris per pasangan:", test.groupby(["movie_title", "cinema_ids"]).size().value_counts().to_dict())

### Cinemas and cities
Size of cinemas and cities by average tickets per film-day, and how many cinemas each city has.

In [ ]:
cluster_size = train.groupby("cinema_ids")["total_ticket"].mean()
city_size = train.groupby("city_name")["total_ticket"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(np.log1p(cluster_size), bins=30)
ax[0].set_title("log1p(rata-rata tiket) per klaster")
city_size.head(15).plot(kind="bar", ax=ax[1], title="top 15 kota berdasarkan total tiket")
train.groupby("city_name")["cinema_ids"].nunique().value_counts().sort_index().plot(kind="bar", ax=ax[2], title="jumlah kota per banyaknya klaster", xlabel="klaster dalam satu kota")
plt.tight_layout()
plt.show()

### Ticket prices
Ticket price per city for each day type, and how it relates to the city's average tickets.

In [ ]:
price_wide = prices.pivot(index="city_name", columns="price_day", values="ceil")
city_mean = train.groupby("city_name")["total_ticket"].mean()
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
price_wide[["Weekday", "Friday", "Weekend"]].plot(kind="box", ax=ax[0], title="harga tiket per tipe hari")
ax[1].scatter(price_wide["Weekday"].reindex(city_mean.index), city_mean, alpha=0.6)
ax[1].set_xlabel("harga weekday")
ax[1].set_ylabel("rata-rata tiket per film-klaster-hari")
plt.show()
price_wide.describe()

### Film metadata
age_rating and genre counts in movies.csv, then the median D4-D10 to D1-D3 ratio per genre to see if genre changes how fast sales drop.

In [ ]:
meta = movies.assign(genre=movies["genre"].str.split(", ")).explode("genre")
mv_ratio = ratio[list(range(4, 11))].mean(axis=1).groupby(level="movie_title").median().rename("ratio_d4_d10")
genre_ratio = meta.merge(mv_ratio, left_on="original_title", right_index=True).groupby("genre")["ratio_d4_d10"].agg(["median", "size"])

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
movies["age_rating"].value_counts().plot(kind="bar", ax=ax[0], title="age_rating")
meta["genre"].value_counts().head(15).plot(kind="bar", ax=ax[1], title="top 15 genre")
genre_ratio[genre_ratio["size"] >= 5]["median"].sort_values().plot(kind="barh", ax=ax[2], title="median rasio D4-D10 per genre (n>=5)")
plt.tight_layout()
plt.show()

### Occupancy and shows
How tickets relate to the number of shows and occupancy, both direct signals of how many slots a cinema gives the film.

In [ ]:
sample = train.sample(20000, random_state=42)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].scatter(np.log1p(sample["total_show"]), np.log1p(sample["total_ticket"]), s=2, alpha=0.3)
ax[0].set_xlabel("log1p(total_show)")
ax[0].set_ylabel("log1p(total_ticket)")
ax[1].scatter(sample["occupation_rate"], np.log1p(sample["total_ticket"]), s=2, alpha=0.3)
ax[1].set_xlabel("occupation_rate")
plt.show()
train[["total_ticket", "occupation_rate", "total_show"]].corr(method="spearman")

### Incomplete days in train
Days with far fewer rows than the 15-day median are recording gaps, not real drops in demand. Target rows on these days are mostly fake zeros, and pairs whose D1-D3 touches them get a wrong scale.

In [ ]:
rows_per_day = train.groupby("date_show").size().reindex(pd.date_range(train["date_show"].min(), train["date_show"].max()), fill_value=0)
row_share = rows_per_day / rows_per_day.rolling(15, center=True, min_periods=5).median()
GAP_DAYS = row_share.index[row_share < 0.7]
print("gap days:", [d.date().isoformat() for d in GAP_DAYS])
pd.DataFrame({"rows": rows_per_day, "share_of_15d_median": row_share.round(2)}).loc["2025-06-03":"2025-06-20"]

# EDA external datasets

### Calendar coverage
Share of train dates, test_history dates (test D1-D3) and test target dates that fall on each kind of special day. A flag can only be learned if it appears in train.

In [ ]:
NAT_HOL = pd.DatetimeIndex(holidays.loc[holidays["holiday_tipe"] == "holiday", "date"])
CAL_DAYS = {
    "national_holiday": NAT_HOL,
    "cuti_bersama": pd.DatetimeIndex(cuti["date"]),
    "ramadan": pd.DatetimeIndex(ramadan["date"]),
    "school_holiday": pd.DatetimeIndex(school["date"]),
}
DATE_SETS = {"train": train["date_show"], "test_history (D1-D3)": test_hist["date_show"], "test target (D4-D10)": test["date_show"]}

coverage = pd.DataFrame({
    name: {flag: dates.isin(days).mean() for flag, days in CAL_DAYS.items()} for name, dates in DATE_SETS.items()
})
distinct = pd.DataFrame({
    name: {flag: dates[dates.isin(days)].nunique() for flag, days in CAL_DAYS.items()} for name, dates in DATE_SETS.items()
})
print("share of rows:")
print(coverage.round(3))
print("\ndistinct special dates:")
distinct

### Cuti bersama effect on national sales
National tickets on each date divided by the centered 7-day average, grouped by day kind. This shows how much busier a cuti bersama day is than a normal day, next to national holidays and weekends.

In [ ]:
daily = train.groupby("date_show")["total_ticket"].sum()
rel = (daily / daily.rolling(7, center=True).mean()).dropna()
kind = pd.Series(np.select(
    [rel.index.isin(NAT_HOL), rel.index.isin(CAL_DAYS["cuti_bersama"]), rel.index.dayofweek >= 5],
    ["national holiday", "cuti bersama", "weekend"], "weekday"), index=rel.index)
print(rel.groupby(kind).agg(["mean", "median", "count"]).round(2))
rel[kind == "cuti bersama"].round(2).rename("ticket / 7-day mean").to_frame().join(cuti.set_index("date")["name"])

### Cuti bersama and the target ratio
Median target ratio (tickets / pair D1-D3 average) per horizon, split by whether the target date is a cuti bersama day, using the train pairs from the decay curve step.

In [ ]:
pair_long = ratio[list(range(4, 11))].stack().rename("ratio").reset_index()
pair_long.columns = ["movie_title", "cinema_ids", "horizon", "ratio"]
pair_long["date"] = pair_long["movie_title"].map(eda_d1) + pd.to_timedelta(pair_long["horizon"] - 1, unit="D")
pair_long["cuti_bersama"] = pair_long["date"].isin(CAL_DAYS["cuti_bersama"])
pair_long["national_holiday"] = pair_long["date"].isin(NAT_HOL)
print("films with a cuti bersama target date:", pair_long.loc[pair_long["cuti_bersama"], "movie_title"].nunique())
pair_long.pivot_table(index=["cuti_bersama", "national_holiday"], columns="horizon", values="ratio", aggfunc="median").round(3)

### Ramadan in test_history
D1-D3 demand of each test film grouped by when it opened: occupancy, tickets per show and tickets per cinema-day. Films opening during Ramadan are compared with the weeks before it and with the Lebaran releases.

In [ ]:
th = test_hist.copy()
th["d1"] = th["movie_title"].map(th.groupby("movie_title")["date_show"].min())
RAMADAN_START, LEBARAN_RELEASE = ramadan["date"].min(), pd.Timestamp("2026-03-18")
film = th.groupby("movie_title").agg(
    d1=("d1", "first"), occ=("occupation_rate", "mean"),
    tickets=("total_ticket", "sum"), shows=("total_show", "sum"), cinema_days=("total_ticket", "size"),
)
film["tickets_per_show"] = film["tickets"] / film["shows"]
film["tickets_per_cinema_day"] = film["tickets"] / film["cinema_days"]
film["opening"] = np.select(
    [film["d1"] >= LEBARAN_RELEASE, film["d1"] >= RAMADAN_START, film["d1"] >= "2026-01-05"],
    ["4 Lebaran release", "3 during Ramadan", "2 Jan-Feb before Ramadan"], "1 Oct-Dec")
film.groupby("opening")[["occ", "tickets_per_show", "tickets_per_cinema_day"]].agg(["median", "size"]).round(2)

### Test_history daily occupancy
Median occupancy of all D1-D3 rows per day in test_history, with Ramadan shaded and cuti bersama days marked, to see the demand level across the test period.

In [ ]:
occ_day = test_hist.groupby("date_show")["occupation_rate"].median()
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(occ_day.index, occ_day.rolling(7, center=True, min_periods=3).mean(), label="7-day mean of daily median occupancy")
ax.axvspan(ramadan["date"].min(), ramadan["date"].max(), color="green", alpha=0.15, label="Ramadan")
for d in CAL_DAYS["cuti_bersama"][CAL_DAYS["cuti_bersama"] >= test_hist["date_show"].min()]:
    ax.axvline(d, color="orange", alpha=0.7)
for d in NAT_HOL[NAT_HOL >= test_hist["date_show"].min()]:
    ax.axvline(d, color="red", alpha=0.3)
ax.set_ylabel("occupancy")
ax.legend()
plt.show()

### Test target rows by calendar group
How many test target rows fall in each calendar situation, which tells how much each external flag can change the score.

In [ ]:
test_d1_all = test.groupby("movie_title")["date_show"].transform("min") - pd.Timedelta(days=3)
d13_dates = [test_d1_all + pd.Timedelta(days=k) for k in range(3)]
groups = pd.DataFrame({
    "target_cuti_bersama": test["date_show"].isin(CAL_DAYS["cuti_bersama"]),
    "target_ramadan": test["date_show"].isin(CAL_DAYS["ramadan"]),
    "d13_ramadan": np.any([d.isin(CAL_DAYS["ramadan"]) for d in d13_dates], axis=0),
    "target_school": test["date_show"].isin(CAL_DAYS["school_holiday"]),
    "target_national_holiday": test["date_show"].isin(NAT_HOL),
})
groups.agg(["sum", "mean"]).T.round(3)